# 09 - Validacion del agente: nivel N1 (herramientas y verificador)

Rama: `08AgentValidationAndSimulator`. Sigue el protocolo `docx/03Evidencia/08-11-26_ProtocoloValidacionAgente.md`, escrito antes de ejecutar. Este cuaderno **no entrena ni recalibra** nada y **no usa un LLM**: prueba las herramientas del agente y el verificador con estados reales y modificados.

| Parte | Que hace | Publica en Kaggle |
|---|---|---|
| 1 | Elige los casos con la semilla 2026 (test de A, B, C y errores muy seguros) y extrae sus observaciones de las repeticiones publicas | `agente/validacion/casos.csv`, `observaciones.json`, `partidas.json.gz` |
| 2 | Bateria N1: escenarios normales, limite, entradas inesperadas, otras poblaciones, sinteticos y errores del modelo | `agente/validacion/resultados_n1.csv`, `detalle_n1.csv`, `latencias_n1.csv` |
| 3 | Analisis: poblaciones, errores muy seguros y direccion de los cambios | nada |

El nivel N2 (el agente completo con Claude) corre en la PC con `10_validacion_llm`, sobre los mismos casos.

In [2]:
# PREPARACION COMPLETA DEL CUADERNO (rama 08AgentValidationAndSimulator)
# PROPOSITO: dejar el servidor listo: repo en la rama 07, librerias fijadas, tablas del
# dataset del equipo en /content/datos y los archivos del modelo final en /content/resultados.
# ESPERADO: la linea de nucleos/RAM, "Tablas listas | caracteristicas: 62" y
# "Modelo final y metricas de test listos" y el SHA-256 del modelo del agente.
import json
import subprocess
import sys
import time
from pathlib import Path

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "08AgentValidationAndSimulator"
REPO_DIR = Path("/content/repo")
if not REPO_DIR.exists():
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo fetch origin {BRANCH}
    !git -C /content/repo checkout {BRANCH}
    !git -C /content/repo pull --ff-only origin {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for nombre in [m for m in list(sys.modules) if m.split(".")[0] in ("ptcg_wp", "ptcg_agente")]:
    del sys.modules[nombre]

from ptcg_wp.colab import recursos, setup_kaggle
RECURSOS = recursos()
setup_kaggle()

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn import __version__ as sklearn_version
assert sklearn_version == "1.7.2", (f"Se requiere scikit-learn 1.7.2; version cargada: {sklearn_version}. "
                                    "Reinicia el runtime y ejecuta esta celda de nuevo.")
from ptcg_wp import store
from ptcg_wp.features import feature_columns
from ptcg_wp.ingest import load_days
from ptcg_wp.predictor import FICHA_EXPORTADA, MODELO_EXPORTADO
from ptcg_agente import artefactos

DATA_DIR = store.DATOS_DIR
RESULTADOS = store.RESULTADOS_DIR
DATASET = store.load_dataset_id()
store.fetch(DATASET, DATA_DIR, store.expected_files(load_days("configs/dias.yaml")))
schema = pq.ParquetFile(sorted(DATA_DIR.glob("estados_*.parquet"))[0]).schema.names
assert len(feature_columns(schema)) == 62
print("Tablas listas | caracteristicas: 62")

BASE = [MODELO_EXPORTADO, FICHA_EXPORTADA, *artefactos.METRICAS_TEST, artefactos.VEREDICTO]
assert store.listo(BASE, RESULTADOS), "Faltan el modelo final o las metricas de test en Kaggle"
print("Modelo final y metricas de test listos")

COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
DIRTY = bool(subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip())
print("rama:", BRANCH, "| commit:", COMMIT[:7], "| cambios locales:", DIRTY)

from ptcg_agente.modelo import ModeloAgente
from ptcg_agente.registro import Registro

artefactos.preparar(RESULTADOS)
MODELO = ModeloAgente.cargar(RESULTADOS)
REGISTRO = Registro("validacion-n1", MODELO.modelo_sha256, COMMIT[:7],
                    carpeta=RESULTADOS / "agente" / "registros")
print("modelo del agente:", MODELO.ficha["modelo"], "| SHA-256:", MODELO.modelo_sha256[:16])

From https://github.com/Adriel23456/Applied-AI-Agent-Project
 * branch            08AgentValidationAndSimulator -> FETCH_HEAD
Already on '08AgentValidationAndSimulator'
Your branch is up to date with 'origin/08AgentValidationAndSimulator'.
From https://github.com/Adriel23456/Applied-AI-Agent-Project
 * branch            08AgentValidationAndSimulator -> FETCH_HEAD
Already up to date.
/content/repo
Nucleos de CPU: 8 (usables por este proceso: 8) | RAM total 51.0 GiB | disponible 48.4 GiB
Kaggle ya estaba configurado en este kernel
ya existe: estados_2026-06-17.parquet
ya existe: estados_2026-06-18.parquet
ya existe: estados_2026-06-19.parquet
ya existe: estados_2026-06-20.parquet
ya existe: estados_2026-06-22.parquet
ya existe: estados_2026-06-25.parquet
ya existe: estados_2026-07-07.parquet
ya existe: estados_2026-07-11.parquet
ya existe: estados_2026-07-15.parquet
ya existe: estados_2026-07-22.parquet
ya existe: partidas_2026-06-17.parquet
ya existe: partidas_2026-06-18.parquet
ya exis

In [3]:
# PRUEBAS DEL AGENTE Y DE LA VALIDACION
# PROPOSITO: correr las pruebas automaticas del agente, de la bateria de validacion y del
# modelo exportado, con datos sinteticos y sin Kaggle.
# ESPERADO: "N passed" sin "failed" y la linea "OK: pruebas aprobadas".
resultado = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_agente.py",
                            "tests/test_validacion.py", "tests/test_predictor.py"],
                           cwd=REPO_DIR, capture_output=True, text=True)
print(resultado.stdout[-2000:], resultado.stderr[-2000:])
assert resultado.returncode == 0, "fallaron pruebas: revisa la salida"
print("OK: pruebas aprobadas")

....................................                                     [100%]
36 passed in 5.77s
 
OK: pruebas aprobadas


In [ ]:
# PARTE 1 - CASOS DE VALIDACION (agente/validacion/)
# PROPOSITO: elegir los estados del protocolo con la semilla 2026 y extraer sus observaciones de
# las repeticiones publicas de Kaggle (una descarga por partida, unas 150-200 partidas).
# Lee test de A, B y C: ya se usaron en la evaluacion final; aqui no se elige ni ajusta ningun modelo.
# ESPERADO: 90 estados normales de A, 90 de B, 90 de C, los errores muy seguros por etapa y la
# partida mas larga de test de A. Si ya esta en Kaggle, solo se baja.
from ptcg_agente.validacion import casos as cv


def preparar_casos():
    inicio = time.time()
    tabla = cv.tabla_test_a(DATA_DIR, MODELO.predictor)
    print(f"estados utiles de test de A: {len(tabla):,} | partidas: {tabla['game_id'].nunique():,}")
    elegidos = cv.elegir(tabla)
    larga = cv.partida_mas_larga(DATA_DIR, tabla)
    print("partida mas larga:", larga, "| partidas a bajar:", elegidos["game_id"].nunique() + 1)
    cv.extraer(elegidos, larga, RESULTADOS)
    print(f"listo en {(time.time() - inicio) / 60:.1f} min")


store.asegurar([cv.CASOS, cv.OBSERVACIONES, cv.PARTIDAS], preparar_casos,
               "agente: casos de validacion (protocolo 08-11-26)", RESULTADOS)
DATOS = cv.Datos(RESULTADOS)
display(DATOS.casos.groupby(["grupo", "stage"]).size().unstack(fill_value=0)[list(cv.ETAPAS)])
print("repeticiones completas:", len(DATOS.repeticiones), "| partida mas larga:", DATOS.larga)

No estan en Kaggle (3 de 3):
   - agente/validacion/casos.csv
   - agente/validacion/observaciones.json
   - agente/validacion/partidas.json.gz
=> Se entrena.


In [ ]:
# PARTE 2 - BATERIA N1 (agente/validacion/resultados_n1.csv)
# PROPOSITO: correr cada caso del protocolo (secciones 3.1 a 3.6) y guardar si aprobo, fallo o
# es solo registro, el detalle por estado y la latencia de cada herramienta.
# ESPERADO: la tabla por grupo, las metricas de la seccion 5 y una fila por caso. L-10 deberia
# fallar (prediccion del protocolo); M-02 es solo registro.
from ptcg_agente.validacion import bateria, resumen


def correr_bateria():
    resultados, detalle, latencias = bateria.correr_n1(MODELO, DATOS, REGISTRO)
    for tabla, ruta in ((resultados, bateria.RESULTADOS_N1), (detalle, bateria.DETALLE_N1),
                        (latencias, bateria.LATENCIAS_N1)):
        tabla.to_csv(RESULTADOS / ruta, index=False)


store.asegurar([bateria.RESULTADOS_N1, bateria.DETALLE_N1, bateria.LATENCIAS_N1], correr_bateria,
               "agente: bateria de validacion N1", RESULTADOS)
RES = pd.read_csv(RESULTADOS / bateria.RESULTADOS_N1)
DET = pd.read_csv(RESULTADOS / bateria.DETALLE_N1)
LAT = pd.read_csv(RESULTADOS / bateria.LATENCIAS_N1)
display(resumen.por_grupo(RES))
display(resumen.metricas(RES, LAT))
with pd.option_context("display.max_colwidth", 200):
    display(RES[["caso", "aprobado", "obtenido"]])

In [ ]:
# PARTE 3 - ANALISIS
# PROPOSITO: comparar A, B y C; medir cuantos errores muy seguros llegan sin advertencia (M-02,
# fragilidad); y ver la direccion de los cambios en los casos limite L-04 a L-07.
# ESPERADO: tres tablas y la lista de casos fallidos.
DET["acierto"] = (DET["p"] >= 0.5) == DET["y"].astype(bool)
DET["con_advertencia"] = DET["advertencias"].fillna("").ne("")
DET["fuera_de_dominio"] = DET["advertencias"].fillna("").str.contains("fuera_de_dominio")

poblaciones = DET[DET["grupo"].isin(["normal_A", "poblacion_B", "poblacion_C"])]
print("Poblaciones (fraccion por etapa):")
display(poblaciones.groupby(["grupo", "etapa"])[["acierto", "con_advertencia", "fuera_de_dominio"]]
        .mean().round(3).unstack("etapa"))

errores = DET[DET["grupo"].fillna("").str.startswith("error_")]
print("Errores muy seguros: cuantos llegan con alguna advertencia (M-01 y M-02):")
display(errores.groupby(["grupo", "etapa"]).agg(casos=("caso", "size"), con_advertencia=("con_advertencia", "mean"),
                                                p_media=("p", "mean")).round(3))

print("Direccion de los cambios en los casos limite (para la validacion humana):")
with pd.option_context("display.max_colwidth", 200):
    display(RES[RES["caso"].isin(["L-04", "L-05", "L-06", "L-07"])][["caso", "descripcion", "obtenido"]])
fallidos = RES[RES["aprobado"].astype(str) == "False"]
print("Casos fallidos:", fallidos["caso"].tolist() or "ninguno")

## Siguiente paso

- **N2 (agente completo con Claude):** `notebooks/10_validacion_llm.ipynb` en la PC, sobre los mismos casos publicados aqui.
- **Correcciones:** los casos que fallaron se corrigen en esta rama y se vuelven a medir en el cuaderno 10 (que repite N1 localmente) sin cambiar el criterio del protocolo.
- **Informe:** `docx/03Evidencia/DD-MM-AA_ResultadosValidacionAgente.md`.